# Train Baseline `G_SAE` (alpha=0)

This notebook trains a baseline guided sparse autoencoder directly on cached latents and saves concept directions to:

`variables/{dataset}/{model}/{layer}/G_SAE.pth`

The saved artifact is CAV-cache-compatible (`entries.normalized`, `entries.unnormalized`) and includes metadata + model `state_dict`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent
sys.path.append(ROOT.as_posix())

CONFIG = {
    # Naming and cache location
    "name": "G_SAE",
    "dataset": "celeba_attacked",
    "model": "vgg16",
    "layer": "features.29",

    # G_SAE architecture
    "latent_factor": 4.0,
    "topk_ratio": 0.10,
    "direction_source": "decoder",

    # Loss weights (baseline alpha=0)
    "recon_weight": 1.0,
    "cond_weight": 1.0,

    # Optimization
    "seed": 23,
    "batch_size": 16,
    "epochs": 100,
    "learning_rate": 1e-4,
    "weight_decay": 0.0,

    # Debug / runtime controls
    "max_samples": None,  

    # Device
    "device": "mps",

    # Export behavior
    "save_optional_encoder_directions": False,
}

LATENT_CACHE_PATH = ROOT / "variables" / CONFIG["dataset"] / CONFIG["model"] / f"{CONFIG['layer']}.pth"
EXPORT_PATH = ROOT / "variables" / CONFIG["dataset"] / CONFIG["model"] / CONFIG["layer"] / f"{CONFIG['name']}.pth"

print(f"Latent cache: {LATENT_CACHE_PATH}")
print(f"Export path : {EXPORT_PATH}")


Latent cache: /Users/erogullari/Desktop/Workspace/cav-disentanglement/variables/celeba_attacked/vgg16/features.29.pth
Export path : /Users/erogullari/Desktop/Workspace/cav-disentanglement/variables/celeba_attacked/vgg16/features.29/G_SAE.pth


In [17]:
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from cav_models import G_SAE


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_device(device_name: str) -> torch.device:
    if device_name == "cuda" and torch.cuda.is_available():
        return torch.device("cuda")
    if device_name == "mps" and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def normalized_recon_mse(recons: torch.Tensor, x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    per_sample = ((recons - x) ** 2).mean(dim=1) / (x.pow(2).mean(dim=1) + eps)
    return per_sample.mean()


def get_concept_directions(model: G_SAE, source: str = "decoder") -> tuple[torch.Tensor, torch.Tensor]:
    if source == "decoder":
        cavs = model.decoder.weight[:, : model.n_concepts].T.detach().cpu()
        bias = torch.zeros(model.n_concepts, dtype=cavs.dtype)
        return cavs, bias

    if source == "encoder":
        cavs = model.encoder.weight[: model.n_concepts, :].detach().cpu()
        if model.encoder.bias is None:
            bias = torch.zeros(model.n_concepts, dtype=cavs.dtype)
        else:
            bias = model.encoder.bias[: model.n_concepts].detach().cpu()
        return cavs, bias

    raise ValueError(f"Unknown direction source: {source}")


def normalize_cavs_and_bias(cavs: torch.Tensor, bias: torch.Tensor, eps: float = 1e-12) -> tuple[torch.Tensor, torch.Tensor]:
    norms = torch.norm(cavs, dim=1, keepdim=True).clamp_min(eps)
    cavs_norm = cavs / norms
    bias_norm = bias / norms.squeeze(1)
    return cavs_norm, bias_norm


seed_everything(int(CONFIG["seed"]))
device = resolve_device(CONFIG["device"])
print(f"Using device: {device}")


Using device: mps


In [18]:
assert LATENT_CACHE_PATH.exists(), f"Latent cache not found: {LATENT_CACHE_PATH}"

latent_payload = torch.load(LATENT_CACHE_PATH, map_location="cpu", weights_only=True)
encs = latent_payload["encs"].float()
labels = latent_payload["labels"].float().clamp(min=0)

if CONFIG["max_samples"] is not None:
    max_samples = int(CONFIG["max_samples"])
    encs = encs[:max_samples]
    labels = labels[:max_samples]

n_samples, n_features = encs.shape
n_concepts = labels.shape[1]

print(f"encs   shape: {tuple(encs.shape)}")
print(f"labels shape: {tuple(labels.shape)}")

encs   shape: (20438, 512)
labels shape: (20438, 42)


In [19]:
model = G_SAE(
    n_concepts=n_concepts,
    n_features=n_features,
    device=str(device),
    latent_factor=float(CONFIG["latent_factor"]),
    topk_ratio=float(CONFIG["topk_ratio"]),
    recon_weight=float(CONFIG["recon_weight"]),
    cond_weight=float(CONFIG["cond_weight"]),
    direction_source=str(CONFIG["direction_source"]),
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=float(CONFIG["learning_rate"]),
    weight_decay=float(CONFIG["weight_decay"]),
)

dataset = TensorDataset(encs, labels)
loader = DataLoader(
    dataset,
    batch_size=int(CONFIG["batch_size"]),
    shuffle=True,
    drop_last=False,
)

history = {
    "total": [],
    "recon": [],
    "cond": [],
}

In [20]:
# Training loop
for epoch in range(int(CONFIG["epochs"])):
    model.train()
    epoch_total = 0.0
    epoch_recon = 0.0
    epoch_cond = 0.0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad(set_to_none=True)
        _, latents, recons = model(x_batch)

        recon_loss = normalized_recon_mse(recons, x_batch)
        cond_features = latents[:, : model.n_concepts]
        cond_loss = F.binary_cross_entropy(cond_features, y_batch)

        total_loss = (
            float(CONFIG["recon_weight"]) * recon_loss
            + float(CONFIG["cond_weight"]) * cond_loss
        )

        total_loss.backward()
        optimizer.step()

        epoch_total += float(total_loss.detach().cpu())
        epoch_recon += float(recon_loss.detach().cpu())
        epoch_cond += float(cond_loss.detach().cpu())

    n_steps = max(len(loader), 1)
    mean_total = epoch_total / n_steps
    mean_recon = epoch_recon / n_steps
    mean_cond = epoch_cond / n_steps

    history["total"].append(mean_total)
    history["recon"].append(mean_recon)
    history["cond"].append(mean_cond)

    print(
        f"Epoch {epoch + 1:03d}/{int(CONFIG['epochs'])} | "
        f"total={mean_total:.6f} recon={mean_recon:.6f} cond={mean_cond:.6f}"
    )

Epoch 001/100 | total=21.462782 recon=0.273543 cond=21.189239
Epoch 002/100 | total=21.078422 recon=0.170642 cond=20.907780
Epoch 003/100 | total=20.999120 recon=0.140750 cond=20.858369
Epoch 004/100 | total=20.912505 recon=0.122364 cond=20.790141
Epoch 005/100 | total=20.848543 recon=0.109676 cond=20.738867
Epoch 006/100 | total=20.810632 recon=0.100735 cond=20.709897
Epoch 007/100 | total=20.793421 recon=0.094697 cond=20.698724
Epoch 008/100 | total=20.789138 recon=0.090642 cond=20.698495
Epoch 009/100 | total=20.766269 recon=0.087001 cond=20.679268
Epoch 010/100 | total=20.734605 recon=0.083468 cond=20.651137
Epoch 011/100 | total=20.705980 recon=0.080420 cond=20.625560
Epoch 012/100 | total=20.476676 recon=0.078589 cond=20.398087
Epoch 013/100 | total=20.101994 recon=0.077539 cond=20.024455
Epoch 014/100 | total=19.998206 recon=0.076006 cond=19.922200
Epoch 015/100 | total=19.938268 recon=0.073903 cond=19.864365
Epoch 016/100 | total=19.869313 recon=0.071234 cond=19.798078
Epoch 01

In [21]:
model = model.to("cpu")

# Export selected directions (decoder by default).
source = str(CONFIG["direction_source"])
cavs_unnorm, bias_unnorm = get_concept_directions(model, source=source)
cavs_norm, bias_norm = normalize_cavs_and_bias(cavs_unnorm, bias_unnorm)

metadata = {
    "name": str(CONFIG["name"]),
    "dataset": str(CONFIG["dataset"]),
    "model": str(CONFIG["model"]),
    "layer": str(CONFIG["layer"]),
    "direction_source": source,
    "latent_factor": float(CONFIG["latent_factor"]),
    "topk_ratio": float(CONFIG["topk_ratio"]),
    "topk": int(model.topk),
    "n_samples": int(n_samples),
    "n_features": int(model.n_features),
    "n_concepts": int(model.n_concepts),
    "n_latents": int(model.n_latents),
    "seed": int(CONFIG["seed"]),
    "batch_size": int(CONFIG["batch_size"]),
    "epochs": int(CONFIG["epochs"]),
    "learning_rate": float(CONFIG["learning_rate"]),
    "weight_decay": float(CONFIG["weight_decay"]),
    "recon_weight": float(CONFIG["recon_weight"]),
    "cond_weight": float(CONFIG["cond_weight"]),
    "baseline_definition": "guided_sae_alpha_0",
    "history": history,
}

if bool(CONFIG["save_optional_encoder_directions"]):
    enc_cavs_unnorm, enc_bias_unnorm = get_concept_directions(model, source="encoder")
    enc_cavs_norm, enc_bias_norm = normalize_cavs_and_bias(enc_cavs_unnorm, enc_bias_unnorm)
    metadata["optional_encoder_export"] = {
        "available": True,
        "direction_source": "encoder",
        "entries": {
            "normalized": {
                "cavs": enc_cavs_norm,
                "bias": enc_bias_norm,
            },
            "unnormalized": {
                "cavs": enc_cavs_unnorm,
                "bias": enc_bias_unnorm,
            },
        },
    }
else:
    metadata["optional_encoder_export"] = {"available": False}

payload = {
    "type": str(CONFIG["name"]),
    "entries": {
        "normalized": {
            "cavs": cavs_norm,
            "bias": bias_norm,
        },
        "unnormalized": {
            "cavs": cavs_unnorm,
            "bias": bias_unnorm,
        },
    },
    "metadata": metadata,
    "state_dict": model.state_dict(),
}

EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(payload, EXPORT_PATH)

print(f"Saved baseline {CONFIG['name']} artifact to: {EXPORT_PATH}")


Saved baseline G_SAE artifact to: /Users/erogullari/Desktop/Workspace/cav-disentanglement/variables/celeba_attacked/vgg16/features.29/G_SAE.pth


In [22]:
assert EXPORT_PATH.exists(), f"Missing export file: {EXPORT_PATH}"
reloaded = torch.load(EXPORT_PATH, map_location="cpu", weights_only=True)

assert reloaded["type"] == "G_SAE", f"Unexpected type: {reloaded['type']}"

cavs = reloaded["entries"]["normalized"]["cavs"]
bias = reloaded["entries"]["normalized"]["bias"]

assert cavs.shape == (n_concepts, n_features), (
    f"Expected cavs shape {(n_concepts, n_features)}, got {tuple(cavs.shape)}"
)
assert bias.shape[0] == n_concepts, (
    f"Expected bias length {n_concepts}, got {bias.shape[0]}"
)

norms = torch.norm(cavs, dim=1)
assert torch.allclose(norms, torch.ones_like(norms), atol=1e-4), "Normalized cavs are not unit norm"

meta = reloaded["metadata"]
assert meta["name"] == "G_SAE"
assert int(meta["n_features"]) == n_features
assert int(meta["n_concepts"]) == n_concepts

print("Validation passed.")
print(f"normalized.cavs shape: {tuple(cavs.shape)}")
print(f"normalized.bias shape: {tuple(bias.shape)}")
print(f"metadata.name: {meta['name']}")


Validation passed.
normalized.cavs shape: (42, 512)
normalized.bias shape: (42,)
metadata.name: G_SAE
